In [ ]:
import numpy as np
import pickle
from collections import defaultdict

In [ ]:
def get_most_prominent_string(arr):
    unique_values, counts = np.unique(arr, return_counts=True)
    max_index = np.argmax(counts)
    most_prominent = unique_values[max_index]
    most_prominent_count = counts[max_index]
    fraction = most_prominent_count / len(arr)
    return most_prominent, fraction

def memory_categorization(categorized_trajectory, p, window_size):
    """
    Given categorized trajectories via 004_PRC_interaction_chains.ipynb.
    Categorizes them further to account for memory effects. 
    """
    new_categorized = np.copy(categorized_trajectory)
    fg_types=['Nup2', 'Nsp1', 'Nup100', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
    n_chains_per_fg=[16, 48, 16, 16, 32, 32, 16, 8, 16]
    for i in range(window_size, len(categorized_trajectory)):
        most_prominent, fraction = get_most_prominent_string(categorized_trajectory[i-window_size:i])
        if fraction > p:
            new_categorized[i] = categorized_trajectory + "_mem"

def 

In [ ]:
# Markov model given new categorization

def normalize_rows(counts_matrix):
    transition_matrix = np.zeros_like(counts_matrix)
    n_states = counts_matrix.shape[0]
    for i in range(n_states):
        row_sum = np.sum(counts_matrix[i, :])
        if row_sum == 0:
            print(f"Row sum is 0, setting row {i} to 0    :( ")
            transition_matrix[i, :] = 0
            continue
        transition_matrix[i, :] = counts_matrix[i, :] / row_sum
    return transition_matrix

def generate_counts_matrix(data, fg_types, n_chains_per_fg):
    # Get unique states
    states = []
    for i, fg_type in enumerate(fg_types):
        for j in range(n_chains_per_fg[i]):
            states.append(f"{fg_type}_{j:02d}")
    for i, fg_type in enumerate(fg_types):
        for j in range(n_chains_per_fg[i]):
            states.append(f"{fg_type}_{j:02d}_mem")
    states = sorted(states)
    states.append("nuc")
    states.append("cyt")
    n_states = len(states)
    
    # Create state to index mapping
    state_to_idx = {state: idx for idx, state in enumerate(states)}
    
    # Initialize transition counts
    counts = defaultdict(lambda: defaultdict(int))
    
    # Count transitions
    for sequence in data:
        for t in range(len(sequence) - 1):
            current_state = sequence[t]
            next_state = sequence[t + 1]
            counts[current_state][next_state] += 1
    return counts, states

def eightwise_symmetrize(data):
    """Given a matrix, makes it 8-wise symmetric"""
    if (data.shape[0] % 8 != 0) or (data.shape[1] % 8 != 0):
        raise Exception("invalid matrix shape")
    mask_base = np.array(np.eye(8, dtype=bool))
    masks = [np.roll(mask_base, shift=i, axis=0) for i in range(8)]
    new_data = np.zeros_like(data)
    for mask in masks:
        for i in range(int(data.shape[0] / 8)):
            for j in range(int(data.shape[1] / 8)):
                new_data[i*8:(i+1)*8, j*8:(j+1)*8][mask] = np.mean(data[i*8:(i+1)*8, j*8:(j+1)*8][mask])
    return new_data

def eightwise_symmetrize_vec(data):
    if data.ndim != 1:
        raise Exception("not a vector")
    if data.shape[0] % 8 != 0:
        raise Exception("invalid vector shape")
    for i in range(int(data.shape[0] / 8)):
        data[i*8:(i+1)*8] = np.mean(data[i*8:(i+1)*8])
    return data

def generate_transition_matrix(data, fg_types, n_chains_per_fg, init_1 = False, symmetrize = False, add_to_diag = 0):
    """
    Generate a transition matrix from an array of categorical time series.
    
    Parameters:
    -----------
    data : array-like
        Array of shape [n_diffusers, t] containing categorical data (strings)
        
    Returns:
    --------
    transition_matrix : numpy.ndarray
        Matrix of transition probabilities
    states : list
        List of unique states (categories)
    """
    counts, states = generate_counts_matrix(data, fg_types, n_chains_per_fg)
    n_states = len(states)
    
    # Create transition matrix
    if init_1: transition_matrix = np.ones((n_states, n_states))
    else: transition_matrix = np.zeros((n_states, n_states))
    transition_matrix += add_to_diag * np.eye(n_states)
    
    # Fill transition matrix with probabilities
    for i, state_i in enumerate(states):
        for j, state_j in enumerate(states):
            transition_matrix[i, j] += counts[state_i][state_j]
    if symmetrize:
        transition_matrix[:-2, -1] = eightwise_symmetrize_vec(transition_matrix[:-2, -1])
        transition_matrix[:-2, -2] = eightwise_symmetrize_vec(transition_matrix[:-2, -2])
        transition_matrix[-1, :-2] = eightwise_symmetrize_vec(transition_matrix[-1, :-2])
        transition_matrix[-2, :-2] = eightwise_symmetrize_vec(transition_matrix[-2, :-2])
        transition_matrix[:-2, :-2] = eightwise_symmetrize(transition_matrix[:-2, :-2])
    transition_matrix = normalize_rows(transition_matrix)
    
    return transition_matrix, states

In [ ]:
with open('data/categorized/interactions-150-180-100ns.pickle', 'rb') as f:
    categorized_trajectories = pickle.load(f)
    
categorized_trajectories = memory_categorization(categorized_trajectories, 0.4, 10)

with open('data/categorized/interactionsmem-150-180-100ns-.pickle', 'wb') as f:
    pickle.dump(categorized_trajectories, f)

tm, states = generate_transition_matrix(categorized_trajectories,
                                        fg_types=['Nup2', 'Nsp1', 'Nup100', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
                                        n_chains_per_fg=[16, 48, 16, 16, 16, 32, 32, 16, 8, 16],
                                        init_1 = True,
                                        symmetrize = True,
                                        add_to_diag = 0)

with open('data/transition_matrices/interactionsmem-150-180-sym-100ns.pickle', 'wb') as f:
    pickle.dump((tm, states), f)